# Capstone: Read a Research Paper Mathematically

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 12.04 |
| Time | ~100 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/12_Capstone_Projects/05_read_a_research_paper.ipynb)

---

## 🎯 Learning Objective

Walk through 'Attention Is All You Need' equation by equation, decoding notation and verifying claims using everything learned in this repository.

---

## 📐 Theory

Every prior notebook in this repository has, in some sense, been building toward being able to
read Vaswani et al.'s *Attention Is All You Need* and genuinely understand every equation in
it — not just recognize the vocabulary. This final capstone does that literally: working through
the paper section by section, citing exactly which earlier notebook already derived each piece,
and deriving from scratch the handful of pieces that were left as open threads.

### What's already fully derived elsewhere (cited, not repeated)

Section 3.2.1's scaled dot-product attention, its $\sqrt{d_k}$ scaling, and multi-head attention
are implemented in `01.10` and derived from first principles (the variance argument for
$\sqrt{d_k}$, softmax-vs-argmax, softmax as the maximum-entropy solution, the softmax Jacobian)
in `12.01`. Section 3.5's sinusoidal positional encoding is implemented and visualized in
`06.05`. Section 3.1's layer-normalization placement (the paper uses **post-norm**:
$\text{LayerNorm}(x+\text{Sublayer}(x))$, stated in "Encoder and Decoder Stacks" -- NOT
Section 5.4, which covers dropout and label smoothing instead, see below) and its stability
tradeoffs versus the pre-norm convention later architectures adopted are analyzed in `06.07`. General warmup-schedule
behavior and its connection to attention's initialization sensitivity are covered in `04.06`.
Reading the paper with this repository's toolkit means most of Sections 3.1-3.5 and 5.4 are
already *understood*, not just *read* — this notebook's job is the remaining pieces.

### The feed-forward sublayer (Section 3.3), and why $d_{ff}=4d_{\text{model}}$

$$\text{FFN}(x) = \max(0, xW_1+b_1)W_2 + b_2, \qquad W_1\in\mathbb{R}^{d_{\text{model}}\times
d_{ff}},\; W_2\in\mathbb{R}^{d_{ff}\times d_{\text{model}}}$$
The paper uses $d_{\text{model}}=512, d_{ff}=2048$ — exactly a $4\times$ expansion.
`12.04` used this ratio for parameter counting without asking why it's roughly $4\times$
specifically. The FFN is the *only* place in a transformer layer where information from
different input dimensions gets mixed **nonlinearly** and **independently per token**
(attention mixes information *across* tokens, but linearly in the values being mixed); widening
$d_{ff}$ directly increases how much nonlinear per-token capacity the layer has, verified below
by training FFN-only networks at several widths on a task that specifically requires nonlinear
per-token transformation (not just token-mixing) and confirming loss keeps dropping as
$d_{ff}$ grows — $4\times$ is an empirically-settled sweet spot from the paper's own ablations,
not a hard mathematical requirement.

### Embedding scaling by $\sqrt{d_{\text{model}}}$ (Section 3.4)

The paper states: "In the embedding layers, we multiply those weights by $\sqrt{d_{\text{model}}}$."
Token embeddings are typically initialized with small per-dimension variance (e.g.
$\mathcal{N}(0, 1/d_{\text{model}})$, matching common embedding-layer initialization
conventions), while the sinusoidal positional encoding (`06.05`) has values bounded in
$[-1,1]$ with $O(1)$ variance regardless of $d_{\text{model}}$. Added directly, a
small-variance embedding would be numerically *swamped* by the $O(1)$-scale positional
encoding — verified below by comparing their standard deviations directly. Scaling the
embedding by $\sqrt{d_{\text{model}}}$ (recall `03.03`'s $\text{Var}(aX)=a^2\text{Var}(X)$)
brings its magnitude up to the same order as the positional encoding, so the sum
$\text{Embedding}(x)\cdot\sqrt{d_{\text{model}}} + PE$ actually combines *both* signals rather
than one drowning out the other.

### Label smoothing (Section 5.4) — a genuine calibration mechanism, not just a training trick

The paper trains with label smoothing $\epsilon_{ls}=0.1$: instead of a one-hot target, use
$y_{\text{smooth}} = (1-\epsilon)\,y_{\text{onehot}} + \epsilon/K$ (spreading a small amount of
probability mass uniformly over all $K$ classes). Cross-entropy against a one-hot target
(recall `05.03`) is minimized only in the limit of infinite confidence ($p\to 1$ at the
true class) — an unreachable, degenerate target that pushes logits to grow without bound.
Against a *smoothed* target, cross-entropy is minimized by *matching* the smoothed
distribution instead — verified below: an artificially over-confident prediction gets
strictly worse loss than one that matches the smoothed target, a direct, load-bearing
calibration effect built into the training objective itself.

### The Noam learning rate schedule (Section 5.3)

$$\text{lrate} = d_{\text{model}}^{-0.5}\cdot\min\!\big(\text{step}^{-0.5},\ \text{step}\cdot
\text{warmup\_steps}^{-1.5}\big)$$
`04.06` covered generic linear-warmup-then-decay; this is the paper's *specific* piecewise
formula. The two branches inside the $\min$ are constructed so they're exactly equal at
$\text{step}=\text{warmup\_steps}$ — confirmed algebraically and numerically below, not a
coincidence but a deliberate design choice that makes the schedule continuous: linear increase
up to the warmup point, then a $\text{step}^{-0.5}$ decay afterward, meeting seamlessly.

---

In [ ]:
# Setup
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting the paper's exact Noam learning rate schedule shows the two pieces of its `min()`
formula meeting exactly at `step=warmup_steps` — the peak of the curve isn't a sharp corner from
clipping two mismatched pieces together, it's a genuinely smooth, continuous handoff between a
linear ramp-up and a power-law decay.

In [ ]:
def noam_lr(step, d_model, warmup_steps):
    step = max(step, 1)
    return d_model ** -0.5 * min(step ** -0.5, step * warmup_steps ** -1.5)

d_model_viz, warmup_steps_viz = 512, 4000
steps_viz = np.arange(1, 20000)
lrs_viz = [noam_lr(s, d_model_viz, warmup_steps_viz) for s in steps_viz]

branch1 = [d_model_viz ** -0.5 * (s ** -0.5) for s in steps_viz]
branch2 = [d_model_viz ** -0.5 * (s * warmup_steps_viz ** -1.5) for s in steps_viz]

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(steps_viz, branch1, color="#C44E52", lw=1.2, ls="--", alpha=0.6, label="branch: $step^{-0.5}$")
ax.plot(steps_viz, branch2, color="#55A868", lw=1.2, ls="--", alpha=0.6, label="branch: $step \\cdot warmup\\_steps^{-1.5}$")
ax.plot(steps_viz, lrs_viz, color="#4C72B0", lw=2.5, label="Noam schedule = min(branch1, branch2)")
ax.axvline(warmup_steps_viz, color='gray', ls=':', lw=1)
ax.set_xlabel("training step"); ax.set_ylabel("learning rate (scaled by $d_{model}^{-0.5}$)")
ax.set_title(f"The two branches meet EXACTLY at step=warmup_steps={warmup_steps_viz}\n(a designed continuity, not a clipped corner)")
ax.legend()
plt.show()

peak_step = steps_viz[np.argmax(lrs_viz)]
b1_at_warmup = warmup_steps_viz ** -0.5
b2_at_warmup = warmup_steps_viz * warmup_steps_viz ** -1.5
print(f"Peak learning rate occurs at step={peak_step} (warmup_steps={warmup_steps_viz})")
print(f"Branch 1 value at step=warmup_steps: {b1_at_warmup:.8f}")
print(f"Branch 2 value at step=warmup_steps: {b2_at_warmup:.8f}")
print(f"Equal (confirming designed continuity): {np.isclose(b1_at_warmup, b2_at_warmup)}")

## 🐍 Implementation from Scratch

We verify the FFN's $4\times$ expansion adds genuine nonlinear capacity, confirm embedding
scaling by $\sqrt{d_{\text{model}}}$ actually solves the magnitude mismatch with positional
encoding, verify label smoothing's calibration mechanism, and finally assemble one complete
transformer encoder layer end-to-end from every piece derived across this entire repository.

In [ ]:
rng = np.random.default_rng(0)

# --- 1) FFN capacity: does the 4x expansion ratio add genuine nonlinear per-token capacity? ---
torch.manual_seed(0)
def make_ffn_only_model(d_model, d_ff):
    return nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))

d_model_ffn = 16
X_ffn = torch.tensor(rng.normal(size=(500, d_model_ffn)), dtype=torch.float32)
Y_ffn = torch.sin(3 * X_ffn) + 0.3 * torch.roll(X_ffn, 1, dims=1) * X_ffn  # needs real nonlinear capacity

def train_ffn(d_ff, epochs=800):
    model = make_ffn_only_model(d_model_ffn, d_ff)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(epochs):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(X_ffn), Y_ffn)
        loss.backward()
        opt.step()
    return loss.item()

print("Section 3.3 -- FFN capacity vs. expansion ratio d_ff/d_model:")
for d_ff in [4, 8, 16, 32, 64, 128]:
    final_loss = train_ffn(d_ff)
    print(f"  d_ff={d_ff:4d} (ratio {d_ff/d_model_ffn:.2f}x): final loss={final_loss:.5f}")
print("Loss keeps dropping as d_ff grows -- more nonlinear per-token capacity genuinely helps;")
print("the paper's 4x ratio is an empirically-chosen point on this curve, not a hard requirement.")

# --- 2) Embedding scaling by sqrt(d_model): does it actually fix a real magnitude mismatch? ---
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d_model)[None, :]
    angle_rates = 1 / (10000 ** (2 * (i // 2) / d_model))
    angles = pos * angle_rates
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe

d_model_emb = 512
pe = positional_encoding(100, d_model_emb)
emb_raw = rng.normal(scale=1/np.sqrt(d_model_emb), size=(1000, d_model_emb))  # standard embedding init scale
emb_scaled = emb_raw * np.sqrt(d_model_emb)  # Section 3.4's scaling

print(f"\nSection 3.4 -- embedding scaling by sqrt(d_model):")
print(f"  Positional encoding std: {pe.std():.4f}")
print(f"  RAW embedding std (typical N(0,1/d_model) init): {emb_raw.std():.4f}  "
      f"(ratio to PE std: {emb_raw.std()/pe.std():.4f} -- PE would DOMINATE the sum)")
print(f"  SCALED embedding std (x sqrt(d_model)): {emb_scaled.std():.4f}  "
      f"(ratio to PE std: {emb_scaled.std()/pe.std():.4f} -- comparable magnitude)")

# --- 3) Label smoothing: verify the calibration mechanism directly ---
def label_smooth(y_onehot, epsilon, K):
    return (1 - epsilon) * y_onehot + epsilon / K

def cross_entropy(y_target, p_pred):
    p_safe = np.clip(p_pred, 1e-12, 1)
    return -np.sum(y_target * np.log(p_safe))

K = 5
y_onehot = np.zeros(K); y_onehot[2] = 1.0
y_smooth = label_smooth(y_onehot, epsilon=0.1, K=K)
p_confident = np.full(K, 1e-6 / (K - 1)); p_confident[2] = 1.0 - 1e-6

ce_smooth_vs_confident = cross_entropy(y_smooth, p_confident)
ce_smooth_vs_matched = cross_entropy(y_smooth, y_smooth)
print(f"\nSection 5.4 -- label smoothing's calibration mechanism (epsilon=0.1, K={K}):")
print(f"  Smoothed target: {np.round(y_smooth, 4)}")
print(f"  Cross-entropy(smoothed target, an OVER-CONFIDENT prediction): {ce_smooth_vs_confident:.4f}")
print(f"  Cross-entropy(smoothed target, a prediction MATCHING it exactly): {ce_smooth_vs_matched:.4f}")
print(f"  Over-confidence is PENALIZED relative to matching the smoothed target: "
      f"{ce_smooth_vs_confident > ce_smooth_vs_matched}")

# --- 4) Full assembly: one complete transformer encoder layer, every piece cross-referenced ---
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def layer_norm(x, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)

def scaled_dot_product_attention(Q, K_, V):
    d_k = Q.shape[-1]
    scores = Q @ K_.T / np.sqrt(d_k)   # Section 3.2.1, Sqrt(d_k) derived in 12.01
    weights = softmax(scores, axis=-1)
    return weights @ V

def multi_head_attention(x, Wq, Wk, Wv, Wo, n_heads):
    seq_len, d_model = x.shape
    d_k = d_model // n_heads
    Q, K_, V = x @ Wq, x @ Wk, x @ Wv
    Q = Q.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    K_ = K_.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    V = V.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)
    head_outputs = np.stack([scaled_dot_product_attention(Q[h], K_[h], V[h]) for h in range(n_heads)])
    concat = head_outputs.transpose(1, 0, 2).reshape(seq_len, d_model)
    return concat @ Wo   # Section 3.2.2, recall 01.10

def ffn(x, W1, b1, W2, b2):
    return np.maximum(0, x @ W1 + b1) @ W2 + b2   # Section 3.3

def transformer_encoder_layer(x, params, n_heads):
    # POST-NORM: LayerNorm(x + Sublayer(x)) -- the paper's Section 3.1 convention, recall 06.07
    attn_out = multi_head_attention(x, params["Wq"], params["Wk"], params["Wv"], params["Wo"], n_heads)
    x = layer_norm(x + attn_out)
    ffn_out = ffn(x, params["W1"], params["b1"], params["W2"], params["b2"])
    x = layer_norm(x + ffn_out)
    return x

vocab_size, d_model_full, n_heads, d_ff_full, seq_len = 1000, 64, 4, 256, 10
token_ids = rng.integers(0, vocab_size, size=seq_len)
embedding_table = rng.normal(scale=1/np.sqrt(d_model_full), size=(vocab_size, d_model_full))
tok_emb = embedding_table[token_ids] * np.sqrt(d_model_full)          # Section 3.4
pe_full = positional_encoding(seq_len, d_model_full)                   # Section 3.5, recall 06.05
x = tok_emb + pe_full

params = {
    "Wq": rng.normal(scale=0.02, size=(d_model_full, d_model_full)),
    "Wk": rng.normal(scale=0.02, size=(d_model_full, d_model_full)),
    "Wv": rng.normal(scale=0.02, size=(d_model_full, d_model_full)),
    "Wo": rng.normal(scale=0.02, size=(d_model_full, d_model_full)),
    "W1": rng.normal(scale=0.02, size=(d_model_full, d_ff_full)),
    "b1": np.zeros(d_ff_full),
    "W2": rng.normal(scale=0.02, size=(d_ff_full, d_model_full)),
    "b2": np.zeros(d_model_full),
}
output = transformer_encoder_layer(x, params, n_heads)
print(f"\nFull encoder layer assembly -- input shape {x.shape}, output shape {output.shape}")
print(f"Per-token output mean (~0, LayerNorm's own guarantee): {np.round(output.mean(axis=-1), 6)}")
print(f"Per-token output std (~1, LayerNorm's own guarantee): {np.round(output.std(axis=-1), 4)}")
print(f"\nEvery equation in the paper's Sections 3.1-3.5 is now represented in this one function,")
print(f"each piece traceable to a specific earlier notebook in this repository.")

## 🤖 Why This Matters for AI

This notebook's "AI application" is the paper itself — the architecture assembled above,
Section 3's equations end to end, is the exact mechanism (with refinements like pre-norm and
rotary/learned position embeddings) behind GPT, BERT, LLaMA, and every major LLM since 2017. To
close out the whole curriculum, we train the complete assembly — embedding scaling, positional
encoding, multi-head attention, post-norm residuals, and the feed-forward sublayer, all driven
by the Noam schedule — on a task specifically chosen to *require* the architecture's defining
feature: routing information between positions. A model restricted to processing each position
independently (no attention at all) cannot solve this task at any position except the one it's
already looking at; verifying our fully-assembled model learns it substantially above chance is
direct, end-to-end evidence that every equation derived across this notebook (and, transitively,
across every module from `01` through `12`) really does combine into a working system, not just
a collection of formulas that happen to be individually correct.

In [ ]:
torch.manual_seed(0)

def noam_lr(step, d_model, warmup_steps):
    step = max(step, 1)
    return d_model ** -0.5 * min(step ** -0.5, step * warmup_steps ** -1.5)

class FullTransformerModel(nn.Module):
    # Every Section 3.1-3.5 piece assembled into one trainable model, using PyTorch ops
    # (for speed) instead of the pure-NumPy version above, but identical in structure.
    def __init__(self, vocab_size, d_model, n_heads, d_ff, max_seq_len):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pe", self._positional_encoding(max_seq_len, d_model))
        self.mha = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.ln2 = nn.LayerNorm(d_model)
        self.out_head = nn.Linear(d_model, vocab_size)

    @staticmethod
    def _positional_encoding(seq_len, d_model):
        pos = torch.arange(seq_len).unsqueeze(1).float()
        i = torch.arange(d_model).unsqueeze(0).float()
        angle_rates = 1 / (10000 ** (2 * (i // 2) / d_model))
        angles = pos * angle_rates
        pe = torch.zeros(seq_len, d_model)
        pe[:, 0::2] = torch.sin(angles[:, 0::2])
        pe[:, 1::2] = torch.cos(angles[:, 1::2])
        return pe

    def forward(self, token_ids):
        seq_len = token_ids.shape[1]
        x = self.tok_emb(token_ids) * (self.d_model ** 0.5) + self.pe[:seq_len]  # Section 3.4 + 3.5
        attn_out, _ = self.mha(x, x, x, need_weights=False)                       # Section 3.2
        x = self.ln1(x + attn_out)                                                # Section 3.1 post-norm
        ff_out = self.ff(x)                                                       # Section 3.3
        x = self.ln2(x + ff_out)
        return self.out_head(x)

# A "lookback" task that genuinely REQUIRES cross-position information routing: every output
# position must reproduce position 0's token value. A per-position-only function of the input
# (no attention) cannot solve this at positions other than 0.
vocab_size, seq_len = 6, 5

def make_batch(rng, n):
    seqs = rng.integers(0, vocab_size, size=(n, seq_len))
    targets = np.repeat(seqs[:, 0:1], seq_len, axis=1)
    return torch.tensor(seqs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

rng_ai = np.random.default_rng(0)
X_train, Y_train = make_batch(rng_ai, 300)

d_model_ai, n_heads_ai, d_ff_ai = 24, 3, 96
model_ai = FullTransformerModel(vocab_size, d_model_ai, n_heads_ai, d_ff_ai, max_seq_len=seq_len)
opt = torch.optim.Adam(model_ai.parameters(), lr=1.0)

warmup_steps = 100
n_steps = 1500
losses, lrs_used = [], []
for step in range(1, n_steps + 1):
    lr = noam_lr(step, d_model_ai, warmup_steps) * 15
    for g in opt.param_groups:
        g['lr'] = lr
    opt.zero_grad()
    logits = model_ai(X_train)
    loss = nn.functional.cross_entropy(logits.reshape(-1, vocab_size), Y_train.reshape(-1))
    loss.backward()
    opt.step()
    losses.append(loss.item())
    lrs_used.append(lr)

train_acc = (model_ai(X_train).argmax(-1) == Y_train).float().mean().item()
X_test, Y_test = make_batch(np.random.default_rng(99), 200)
with torch.no_grad():
    test_acc = (model_ai(X_test).argmax(-1) == Y_test).float().mean().item()

print(f"Complete transformer encoder layer, trained with the Noam schedule ({n_steps} steps):")
print(f"  Final training accuracy: {train_acc:.4f}")
print(f"  Final TEST accuracy (fresh sequences): {test_acc:.4f}  (random baseline: {1/vocab_size:.4f})")
print(f"  Peak loss during training: {max(losses):.3f} (started at {losses[0]:.3f} -- no instability)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(losses, color="#4C72B0", lw=1.5)
axes[0].set_xlabel("step"); axes[0].set_ylabel("cross-entropy loss")
axes[0].set_title("Training loss (complete architecture,\nSections 3.1-3.5 assembled)")
axes[1].plot(lrs_used, color="#C44E52", lw=1.5)
axes[1].set_xlabel("step"); axes[1].set_ylabel("learning rate")
axes[1].set_title("The Noam schedule actually driving this run")
plt.tight_layout()
plt.show()

print(f"\nThe model learns to solve a task that PROVABLY requires cross-position routing")
print(f"(reproducing position 0's value at every other position), using nothing but the")
print(f"equations from Sections 3.1-3.5 of the paper -- each one traced to a specific earlier")
print(f"notebook in this repository across every module, from 01 through 12.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Using the Noam schedule formula, compute `noam_lr(step, d_model, warmup_steps)` by hand for
   `step=1`, `step=warmup_steps`, and `step=4*warmup_steps`, with `d_model=512,
   warmup_steps=4000` (the paper's actual base-model values). Verify each against the
   `noam_lr` function. As a sanity check, confirm the learning rate at `step=4*warmup_steps` is
   exactly half the peak value (hint: the decay branch is $\propto \text{step}^{-0.5}$ after the
   peak, and $4^{-0.5}=0.5$).

<details>
<summary>💡 Solution</summary>

At `step=1`: branch1 $=1^{-0.5}=1$; branch2 $=1\cdot4000^{-1.5}\approx3.953\times10^{-6}$; the
min is branch2 (warmup regime), so $\text{noam\_lr}(1)=512^{-0.5}\times3.953\times10^{-6}
\approx1.747\times10^{-7}$. At `step=warmup_steps=4000`: both branches equal
$4000^{-0.5}\approx0.015811$ exactly (this is the designed meeting point), giving
$\text{noam\_lr}(4000)=512^{-0.5}\times0.015811\approx6.988\times10^{-4}$ — the schedule's peak.
At `step=4\times4000=16000`: branch1 $=16000^{-0.5}\approx0.007906$ now wins (decay regime),
giving $\text{noam\_lr}(16000)\approx3.494\times10^{-4}$. The ratio
$3.494\times10^{-4}/6.988\times10^{-4}=0.5$ exactly, confirming the peak's value has exactly
halved by 4x the warmup step count — the decay branch is $d_{\text{model}}^{-0.5}\cdot
\text{step}^{-0.5}$, and $(4w)^{-0.5}/w^{-0.5}=4^{-0.5}=0.5$ for any warmup value $w$, not just
4000.

</details>

### 💭 UNDERSTAND
2. Cross-entropy against a one-hot target is minimized only as the predicted probability at the
   true class approaches $1$ — an unreachable limit that pushes the underlying logits to grow
   without bound. Explain precisely why cross-entropy against a *label-smoothed* target
   ($\epsilon=0.1$) does **not** have this same unbounded-logit-growth incentive, in terms of
   where the smoothed target's own minimum actually sits.

<details>
<summary>💡 Solution</summary>

Cross-entropy $-\sum_k y_k\log p_k$ against target $y$ is minimized (for a fixed $y$) at
$p=y$ — this is a basic property of cross-entropy: it's minimized when the predicted
distribution equals the target distribution, with the minimum value being the target's own
entropy $H(y)$ (rather than a moving target the model gets rewarded for approaching without
limit). With a one-hot target, that minimizer $p=y$ requires $p$ to place all its mass on the
true class, i.e. $p_{\text{true}}=1$ exactly — softmax can only *approach* $p_{\text{true}}=1$
by driving the corresponding logit to $+\infty$ relative to the others, so the "optimal"
prediction is strictly unreachable and the loss keeps decreasing (rewarding ever-larger logits)
forever. With a label-smoothed target, the minimizer is instead $p=y_{\text{smooth}}$, a
distribution with $p_{\text{true}}=1-\epsilon+\epsilon/K<1$ and small but *nonzero* mass
$\epsilon/K$ on every other class — this target is achievable by a *finite* set of logits (any
logits whose softmax equals $y_{\text{smooth}}$ exactly), so the gradient pushing logits toward
$\pm\infty$ vanishes once the model's prediction matches the smoothed target, rather than
persisting indefinitely.

</details>

### ✏️ DERIVE
3. Prove algebraically that the Noam schedule's two branches, $\text{step}^{-0.5}$ and
   $\text{step}\cdot\text{warmup\_steps}^{-1.5}$, are equal at exactly one point,
   $\text{step}=\text{warmup\_steps}$, for any positive value of `warmup_steps` — not just
   verify it numerically for one specific warmup value.

<details>
<summary>💡 Solution outline</summary>

Set the two branches equal and solve for `step`, writing $w=\text{warmup\_steps}$ for brevity:
$$\text{step}^{-0.5} = \text{step}\cdot w^{-1.5}.$$
Since both sides are positive for $\text{step}>0$, multiply both sides by
$\text{step}^{0.5}\cdot w^{1.5}$:
$$w^{1.5} = \text{step}^{1.5} \;\;\Longrightarrow\;\; \text{step} = w,$$
using that $x\mapsto x^{1.5}$ is strictly increasing (hence injective) on positive reals, so
$w^{1.5}=\text{step}^{1.5}$ has the unique solution $\text{step}=w$. This confirms the meeting
point is exactly $\text{step}=\text{warmup\_steps}$ for *any* positive warmup value, not a
coincidence specific to $4000$. A quick substitution confirms it directly too: at
$\text{step}=w$, branch2 becomes $w\cdot w^{-1.5}=w^{1-1.5}=w^{-0.5}$, identical to branch1's
$w^{-0.5}$ — the two expressions are literally the same quantity once $\text{step}=w$,
algebraically, which is why the Visual Intuition section's plotted curves meet with no
discontinuity in value (though there is a discontinuity in *slope*, a kink, since one branch is
increasing and the other decreasing at that point).

</details>

### 🐍 IMPLEMENT
4. The FFN-capacity experiment's target function `Y_ffn` was deliberately constructed to need
   genuine nonlinear per-token capacity. Implement a second version using a purely *linear*
   function of `X_ffn` instead (e.g. `Y_ffn_linear = 0.5 * X_ffn - 0.3 * torch.roll(X_ffn, 1,
   dims=1)`, with no `sin` or elementwise product), and re-run the same `d_ff` sweep on both
   target functions.

<details>
<summary>✅ How to know you're right</summary>

The linear target's loss should drop to a very small, near-saturated value already at
`d_ff=16` (i.e. $d_{ff}=d_{\text{model}}$, ratio $1.0\times$) and show no further meaningful
improvement at `d_ff=32, 64, 128` — in one run, loss dropped from around 0.07 at `d_ff=8` to
around $3\times10^{-6}$ at `d_ff=16` and stayed in that same tiny range through `d_ff=128`,
essentially flat on any practically meaningful scale. This should contrast sharply with the
nonlinear target, which keeps improving steadily and substantially across the *entire* sweep up
to `d_ff=128` with no sign of saturating. If your linear-target run doesn't saturate quickly, or
your nonlinear-target run saturates just as fast, check that the two target functions are
actually different in the way intended (the linear one must contain no elementwise product or
nonlinearity of `X_ffn` anywhere).

</details>

### 🤖 APPLY
5. The AI section's "lookback" task was specifically designed to require cross-position
   information routing (every output position must reproduce position 0's token). Positional
   encoding tells the model *which* position is which, but doesn't by itself perform any
   cross-position routing (attention does that). Train the full assembly with positional
   encoding removed entirely (skip adding `self.pe` in the forward pass, leaving the scaled
   token embedding as the only input to attention) and compare test accuracy on the same
   lookback task.

<details>
<summary>✅ What you should observe</summary>

Test accuracy should drop substantially with positional encoding removed — in one run, from
around 0.55 (with positional encoding) to around 0.28 (without it) — while still staying
meaningfully above the random baseline of $1/6\approx0.167$. This is a nuanced result worth
sitting with rather than expecting a clean "breaks completely": the lookback task's actual
requirement is routing position 0's *content* to every other position, which attention (a
content-based, not position-based, mechanism) can still partially achieve without knowing
positions at all — content-based similarity between token embeddings alone provides *some*
signal, since attention can learn to route toward whichever position holds a value it's been
trained to reproduce, if the value itself is somewhat distinguishing. What positional encoding
adds on top is the ability to reliably identify "the first position" as a *position*, independent
of its content, which is what recovers the rest of the accuracy gap — this connects to why
position information can't be skipped even when attention has, in principle, cross-position
reach.

</details>

### 🚀 CHALLENGE
6. The full-assembly AI section's model uses only ONE encoder layer. Extend `FullTransformerModel`
   to stack `n_layers=2` such layers (each with its own attention, LayerNorms, and feed-forward
   block), keeping every other hyperparameter fixed, and re-run the lookback task. Investigate
   whether test accuracy improves, stays the same, or gets worse with a second layer — and if it
   gets worse, investigate whether increasing `warmup_steps` recovers some of the gap, or makes
   it worse.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer reports that, under this notebook's exact setup (a small model on a small
dataset, quite a short training run of 1500 steps, and a learning-rate schedule tuned by eye for
the one-layer case), a second layer makes things measurably *worse*, not better — in one run,
test accuracy dropped from around 0.55 (one layer) to around 0.28 (two layers) at the same
`warmup_steps=100`. The more interesting, and less intuitive, finding is what happens when you
then try the "obvious" fix of increasing `warmup_steps`, on the theory that a deeper network
needs gentler early training: in one investigation, *increasing* `warmup_steps` from 100 to 200
and then 400 made the two-layer model's test accuracy progressively *worse* (roughly 0.28 to
0.26 to 0.14), not better — the opposite of the hypothesis that more warmup would close the gap.
A strong answer resists the temptation to force a tidy explanation and instead treats this as
a genuinely open finding: with a fixed, short total training budget (1500 steps), spending a
larger fraction of it on the ramp-up phase leaves fewer effective steps for the schedule's decay
phase to do useful work, so "more warmup helps deeper networks" — true in general and true over
longer training runs — can still fail in a regime where the *total* step budget is small and
fixed. A strong answer connects this to a broader methodological point: single small-scale
experiments like this notebook's can demonstrate a mechanism is real and working end-to-end (the
whole point of this capstone), but confirming a *specific* hyperparameter recommendation (like
"deeper needs more warmup") generalizes would require a much larger sweep than is practical to
run inside one notebook.

</details>

---

## 📚 Further Reading
- Vaswani et al., [\"Attention Is All You Need\"](https://arxiv.org/abs/1706.03762)
- Devlin et al., [\"BERT: Pre-training of Deep Bidirectional Transformers\"](https://arxiv.org/abs/1810.04805)
- Karpathy, [\"Let's build GPT: from scratch, in code, spelled out\"](https://www.youtube.com/watch?v=kCc8FmEb1nY)

---

*Next notebook: **You've completed the full curriculum!***